# Train — SmallUNet (Cervical)

omuro データセット（側面・前屈・後屈 合計324件）を使った頚椎8点ランドマーク検出モデルの訓練。

- モデル: SmallUNet
- ランドマーク: C2_center, C2_ant, C2_post, C7_sup_post, C7_inf_ant, C7_inf_post, T1_ant, T1_post
- sigma: 15, loss: AWL, augment: True
- 出力: `SAVE_DIR/best.pt`, `cervical_best.onnx`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
if [ ! -d spine-measure-assist ]; then
  git clone https://github.com/masaki39/spine-measure-assist.git
fi
cd spine-measure-assist && git pull

In [ ]:
%%bash
pip install uv -q
cd /content/spine-measure-assist
uv sync --extra ml -q

In [ ]:
%%bash
DATA_DIR='/content/drive/MyDrive/spine_data/omuro_cervical'
SAVE_DIR='/content/drive/MyDrive/spine_data/omuro_runs_sigma15'
LANDMARKS='C2_center,C2_ant,C2_post,C7_sup_post,C7_inf_ant,C7_inf_post,T1_ant,T1_post'

mkdir -p "$SAVE_DIR"
cd /content/spine-measure-assist
uv run python train/train.py \
  --data-dir  "$DATA_DIR" \
  --save-dir  "$SAVE_DIR" \
  --landmarks "$LANDMARKS" \
  --backbone  smallunet \
  --sigma     15 \
  --augment   \
  --loss      awl \
  --split-seed 42 \
  --epochs    50 \
  --batch-size 8

In [ ]:
%%bash
SAVE_DIR='/content/drive/MyDrive/spine_data/omuro_runs_sigma15'

cd /content/spine-measure-assist
uv run python train/export_lumbar.py \
  --checkpoint "$SAVE_DIR/best.pt" \
  --output     "$SAVE_DIR/cervical_best.onnx"

In [ ]:
%%bash
DATA_DIR='/content/drive/MyDrive/spine_data/omuro_cervical'
SAVE_DIR='/content/drive/MyDrive/spine_data/omuro_runs_sigma15'

cd /content/spine-measure-assist
uv run python train/eval_cervical.py \
  --model  "$SAVE_DIR/cervical_best.onnx" \
  --dir    "$DATA_DIR" \
  --splits "$SAVE_DIR/splits.json" \
  --subset test